### Data Dependencies

In [8]:
library(rsample)     # data splitting 
library(dplyr)       # data wrangling
library(rpart)       # performing regression trees
library(rpart.plot)  # plotting regression trees
library(ipred)       # bagging
library(caret)       # bagging
library(smotefamily) # smote
library(janitor)     # data cleaning
library(tidyverse) 
library(randomForest)
library(xgboost)

### Cleaning Data 

In [9]:
set.seed(100) # Set seed

# Load & Clean Column Names
merged <- read_csv("data/merged_data.csv", show_col_types = FALSE) |> 
  clean_names()

# Train/test split 

index <- createDataPartition(merged$outbreak, p = 0.75, list = FALSE)
train <- merged[index, ]
test  <- merged[-index, ]

# Drop county and NA columns 
train <- train[, colSums(is.na(train)) == 0]
train <- train[, !names(train) %in% c("county")]

# Ensure all columns are numeric 
non_num <- names(train)[!sapply(train, is.numeric)]
if (length(non_num) > 0) {
  train[non_num] <- lapply(train[non_num], as.numeric)
}

# Mirror same cleaning on test set 
test <- test[, names(test) %in% c(names(train), "outbreak")]

non_num_test <- names(test)[!sapply(test, is.numeric) & names(test) != "outbreak"]
if (length(non_num_test) > 0) {
  test[non_num_test] <- lapply(test[non_num_test], as.numeric)
}

test <- test |>
  mutate(across(
    .cols = -outbreak,
    .fns  = ~ifelse(is.na(.), train_medians[cur_column()], .)
  ))

test$outbreak <- as.numeric(test$outbreak)

stopifnot(sum(is.na(test)) == 0)

cat("Train rows:", nrow(train), "| Cols:", ncol(train), "\n")
cat("Test rows:",  nrow(test),  "| Cols:", ncol(test),  "\n")

# Export to csv
write_csv(train, "data/created/train_clean.csv")
write_csv(test,  "data/created/test_clean.csv")

Train rows: 191 | Cols: 220 
Test rows: 63 | Cols: 220 


### SMOGN (IN PYTHON RUN DIFFERENT KERNEL)

In [10]:
import numpy as np
import pandas as pd
import smogn

# Load cleaned train data from R 
train = pd.read_csv("data/created/train_clean.csv")
randomseed = 100

# Apply SMOGN with manual relevance control points 
# rel_ctrl_pts_rg format: [[x, y, derivative], ...]
# x = outbreak value, y = relevance (0 = common, 1 = rare)
# Set low outbreak counts as common (0) and high counts as rare (1)

outbreak_min = train["outbreak"].min()
outbreak_max = train["outbreak"].max()
outbreak_mid = train["outbreak"].median()
outbreak_nonzero_min = train[train["outbreak"] > 0]["outbreak"].min()

# Ensure control points are strictly increasing and unique
rel_ctrl_pts = [
    [outbreak_min,        0, 0],   # 0 = common
    [outbreak_nonzero_min, 0.5, 0], # first nonzero = starting to get rare
    [outbreak_max,        1, 0],   # max = rarest, oversample most
]

train_balanced = smogn.smoter(
    data            = train,
    y               = "outbreak",
    k               = 5,
    samp_method     = "balance",
    seed            = randomseed,
    rel_thres       = 0.5,
    rel_method      = "manual",
    rel_ctrl_pts_rg = rel_ctrl_pts
)

print("\nOutbreak distribution after SMOGN:")
print(train_balanced["outbreak"].value_counts().sort_index())
print("Train balanced shape:", train_balanced.shape)


# ── 4. Export balanced train set back for R modeling ──────────────────────
train_balanced.to_csv("data/created/train_balanced.csv", index=False)

ERROR: Error in parse(text = input): <text>:1:8: unexpected symbol
1: import numpy
           ^


### Regular Regression Tree (No optimization)

In [3]:
# Read data form CSV
train_balanced <- read_csv("data/created/train_balanced.csv", show_col_types = FALSE) 
test  <- read_csv("data/created/test_clean.csv", show_col_types = FALSE) 

# Train model 
dtree_model <- rpart(outbreak ~ ., data = train_balanced, method = "anova")
#rpart.plot(dtree_model)

# Eval Performance
predictions <- predict(dtree_model, newdata = train)
actual_values <- train$outbreak

# Calculate accuracy metrics using base R math
dtree_rmse <- sqrt(mean((actual_values - predictions)^2))
dtree_mae  <- mean(abs(actual_values - predictions))

rss <- sum((actual_values - predictions)^2)
tss <- sum((actual_values - mean(actual_values))^2)
dtree_r2 <- 1 - (rss / tss)

# Print results to the console
cat("Regression Tree Accuracy Metrics:\n")
cat("RMSE (Root Mean Squared Error):", round(dtree_rmse, 4), "\n")
cat("MAE  (Mean Absolute Error)    :", round(dtree_mae, 4), "\n")
cat("R-squared                     :", round(dtree_r2, 4), "\n")

# Visualize predictions vs actuals
# plot(actual_values, predictions, 
#      main = "Regression Tree: Actual vs. Predicted", 
#      xlab = "Actual Outbreak", ylab = "Predicted Outbreak", 
#      pch = 19, col = "blue")
# abline(0, 1, col = "red", lwd = 2) # Perfect prediction reference line


Regression Tree Accuracy Metrics:
RMSE (Root Mean Squared Error): 7.996 
MAE  (Mean Absolute Error)    : 3.6529 
R-squared                     : -0.08 


### Random Forest (No optimization)

In [4]:
# Train Model
rforest_model <- randomForest(outbreak ~ ., data = train_balanced, ntree = 500, importance = TRUE)

predictions <- predict(rforest_model, newdata = train)
actual_values <- train$outbreak

# Calculate accuracy metrics using base R math
rforest_rmse <- sqrt(mean((actual_values - predictions)^2))
rforest_mae  <- mean(abs(actual_values - predictions))

rss <- sum((actual_values - predictions)^2)
tss <- sum((actual_values - mean(actual_values))^2)
rforest_r2 <- 1 - (rss / tss)

# Print results to the console
cat("Regression Tree Accuracy Metrics:\n")
cat("RMSE (Root Mean Squared Error):", round(rforest_rmse, 4), "\n")
cat("MAE  (Mean Absolute Error)   :", round(rforest_mae, 4), "\n")
cat("R-squared                    :", round(rforest_r2, 4), "\n")

# #Visualize predictions vs actuals
# plot(actual_values, predictions, 
#      main = "Regression Tree (Random Forest): Actual vs. Predicted", 
#      xlab = "Actual Outbreak", ylab = "Predicted Outbreak", 
#      xlim = c(0, 420), ylim = c(0, 420),
#      pch = 19, col = "blue")
# abline(0, 1, col = "red", lwd = 2) # Perfect prediction reference line

Regression Tree Accuracy Metrics:
RMSE (Root Mean Squared Error): 5.0223 
MAE  (Mean Absolute Error)   : 2.6392 
R-squared                    : 0.5739 


### Boosted Tree (No optimization)

In [5]:
# 1. Prepare data
X_train <- as.matrix(train_balanced[, names(train_balanced) != "outbreak"])
y_train <- as.numeric(as.character(train_balanced$outbreak))
dtrain <- xgb.DMatrix(data = X_train, label = y_train)

# Prepare the original data for prediction
# Note: ogtrain should be a DMatrix so xgboost can read it
X_og <- as.matrix(train[, names(train) != "outbreak"])
ogtrain <- xgb.DMatrix(data = X_og) 

# 2. Train the model
boost_model <- xgb.train(
  params = list(
    objective = "reg:squarederror",
    max_depth = 3,
    eta = 0.1
  ),
  data = dtrain,
  nrounds = 500
)

# 3. Predict on the ORIGINAL training set
predictions <- predict(boost_model, ogtrain) # Use dogtrain, not dtrain
actual_values <- as.numeric(as.character(train$outbreak)) # Pull from original 'train' dataframe

# 4. Calculate metrics (dimensions now match)
boost_rmse <- sqrt(mean((actual_values - predictions)^2))
boost_mae <- mean(abs(actual_values - predictions))

rss <- sum((actual_values - predictions)^2)
tss <- sum((actual_values - mean(actual_values))^2)
boost_r2 <- 1 - (rss / tss)

# Print results
cat("Regression Tree Accuracy Metrics (Original Train):\n")
cat("RMSE:", round(boost_rmse, 4), "\n")
cat("MAE :", round(boost_mae, 4), "\n")
cat("R-squared :", round(boost_r2, 4), "\n")

# #Visualize predictions vs actuals
# plot(actual_values, predictions, 
#      main = "Regression Tree (Boosted): Actual vs. Predicted", 
#      xlab = "Actual Outbreak", ylab = "Predicted Outbreak", 
#      xlim = c(0, 420), ylim = c(0, 420),
#      pch = 19, col = "blue")
# abline(0, 1, col = "red", lwd = 2) # Perfect prediction reference line

Regression Tree Accuracy Metrics (Original Train):
RMSE: 3.938 
MAE : 1.6774 
R-squared : 0.738 


# TUNING SECTION

### Setting up all parameters

In [6]:
# classification tree related
maxdepths <- seq(1, 10, by = 1)
# random forest related
mtrys   <- c(seq(10, 200, by=10))
ntrees   <- c(seq(300, 700, by=50))
# XGBoost related
param_grid <- expand.grid( # tuning grid
  max_depth = c(seq(3, 9, by=2)),
  eta = c(0.01,seq(0.05, 0.4, by=0.05))
)

# Folds for cross validation
set.seed(4)

folds <- createFolds(train_balanced$outbreak, k = 10)

### Basic Regression Tree Optimization

In [7]:
set.seed(100)
results    <- data.frame()

for(i in maxdepths){
  fold_rmse <- c()
  fold_mae  <- c()
  fold_r2   <- c()

  for(fold in folds){
    fold_train <- train_balanced[-fold, ]
    fold_test  <- train_balanced[fold, ]

    mod <- rpart(outbreak ~ .,
                 data    = fold_train,
                 method  = "anova",
                 weights = ceiling(fold_train$population),
                 control = rpart.control(maxdepth = i))

    cp_table <- mod$cptable
    opt_cp   <- as.numeric(cp_table[which.min(cp_table[,"xerror"]), "CP"])[1]
    pruned   <- prune(mod, cp = opt_cp)

    predictions   <- predict(pruned, newdata = fold_test)
    actual_values <- fold_test$outbreak

    fold_rmse <- c(fold_rmse, sqrt(mean((actual_values - predictions)^2)))
    fold_mae  <- c(fold_mae,  mean(abs(actual_values - predictions)))

    rss     <- sum((actual_values - predictions)^2)
    tss     <- sum((actual_values - mean(actual_values))^2)
    fold_r2 <- c(fold_r2, 1 - (rss / tss))
  }

  results <- rbind(results, data.frame(
    maxdepth = i,                              
    RMSE     = mean(fold_rmse, na.rm = TRUE),
    MAE      = mean(fold_mae,  na.rm = TRUE),
    R2       = mean(fold_r2,   na.rm = TRUE)
  ))
}

results
best_clas <- results[which.max(results$R2), ]
print(best_clas)

maxdepth,RMSE,MAE,R2
<dbl>,<dbl>,<dbl>,<dbl>
1,14.06202,7.093923,-1.780741
2,14.49831,7.711834,-1.841594
3,15.50914,8.591704,-1.976884
4,15.67277,8.709962,-2.280546
5,15.63755,8.619001,-2.003412
6,14.67126,8.180736,-1.866008
7,15.37925,8.414029,-1.931215
8,14.58389,7.731925,-2.034003
9,15.60875,8.391435,-2.132302


  maxdepth     RMSE      MAE        R2
1        1 14.06202 7.093923 -1.780741


### Random Forest Optimization

In [13]:
results_rf <- data.frame()

set.seed(100)
for(m in mtrys){
  fold_rmse       <- c()
  fold_mae        <- c()
  fold_r2         <- c()
  all_actual      <- c()
  all_predictions <- c()
    
  for(fold in folds){
    fold_train <- train_balanced[-fold, ]
    fold_test  <- train_balanced[fold, ]
      
    rf <- randomForest(outbreak ~ .,
                       data       = fold_train,
                       mtry       = m,
                       ntree      = 500,
                       importance = TRUE)
    
    predictions   <- predict(rf, newdata = fold_test)
    actual_values <- fold_test$outbreak
      
    fold_rmse       <- c(fold_rmse, sqrt(mean((actual_values - predictions)^2)))
    fold_mae        <- c(fold_mae,  mean(abs(actual_values - predictions)))
    rss     <- sum((actual_values - predictions)^2)
    tss     <- sum((actual_values - mean(actual_values))^2)
    fold_r2 <- c(fold_r2, 1 - (rss / tss))
  }
    
  results_rf <- rbind(results_rf, data.frame(
    mtry = m,
    RMSE = mean(fold_rmse, na.rm = TRUE),
    MAE  = mean(fold_mae,  na.rm = TRUE),
    R2   = mean(fold_r2, na.rm = TRUE)
  ))
}

best_rf <- results_rf[which.max(results_rf$R2), ]
print(best_rf)

   mtry     RMSE      MAE         R2
18  180 11.61479 6.657781 -0.2565868


### XGBoost Optimization

In [12]:
results_xgb <- data.frame()

set.seed(100)
for(i in 1:nrow(param_grid)){
  fold_rmse       <- c()
  fold_mae        <- c()
  fold_r2         <- c()
  all_actual      <- c()
  all_predictions <- c()
    
  for(fold in folds){
    fold_train <- train_balanced[-fold, ]
    fold_test  <- train_balanced[fold, ]

    X_fold_train <- as.matrix(fold_train[, names(fold_train) != "outbreak"])
    y_fold_train <- as.numeric(as.character(fold_train$outbreak))
    X_fold_test  <- as.matrix(fold_test[,  names(fold_test)  != "outbreak"])
    y_fold_test  <- as.numeric(as.character(fold_test$outbreak))

    dtrain_fold <- xgb.DMatrix(data = X_fold_train, label = y_fold_train)
    dtest_fold  <- xgb.DMatrix(data = X_fold_test,  label = y_fold_test)

    model <- xgb.train(
      params = list(
        objective   = "reg:squarederror",
        eval_metric = "rmse",
        max_depth   = param_grid$max_depth[i],
        eta         = param_grid$eta[i]
      ),
      data    = dtrain_fold,
      nrounds = 500
    )

    predictions   <- predict(model, newdata = dtest_fold) 
    actual_values <- fold_test$outbreak

    fold_rmse       <- c(fold_rmse, sqrt(mean((actual_values - predictions)^2)))
    fold_mae        <- c(fold_mae,  mean(abs(actual_values - predictions)))
    rss     <- sum((actual_values - predictions)^2)
    tss     <- sum((actual_values - mean(actual_values))^2)
    fold_r2 <- c(fold_r2, 1 - (rss / tss))
  }

  results_xgb <- rbind(results_xgb, data.frame(
    max_depth = param_grid$max_depth[i],
    eta       = param_grid$eta[i],
    RMSE      = mean(fold_rmse, na.rm = TRUE),
    MAE       = mean(fold_mae,  na.rm = TRUE),
    R2        = mean(fold_r2, na.rm = TRUE)
  ))
}

best_xgb <- results_xgb[which.max(results_xgb$R2), ] 
print(best_xgb)

   max_depth eta     RMSE      MAE        R2
25         3 0.3 11.52429 6.215495 0.2050892
